# Large Maze Ellipsoids Diffuser — Training with DeepSet Conditioning + CFG

Trains a VE-SDE score model on the `custom/largemaze_ellipsoids-v1` Minari dataset.  
The model conditions on:
- Start and goal positions
- A **DeepSet-encoded set of up to 5 axis-aligned superellipsoid obstacles** (cx, cy, a, b)

Training uses **Classifier-Free Guidance (CFG)** dropout: with probability `P_UNCOND`,
the ellipsoid conditioning is zeroed out so the model learns both conditional and
unconditional distributions simultaneously.

Sampling uses `dpm_solver_1_cfg_sample` with `guidance_scale` controlling how
strongly the model conditions on the obstacle set.

## 1. Parameters

In [10]:
# ---- Dataset ----
DATASET_ID      = "custom/largemaze_ellipsoids-v1"
LOCAL_DATA_PATH = "../Diffuser/TrajectoryDatasetGeneration/data"  # relative to this notebook
MAX_ELLIPSOIDS  = 5   # fixed set size; missing ellipsoids are padded with zeros

# ---- Model ----
UNET_INPUT_DIM   = 32
DIM_MULTS        = (1, 2, 4)
LOCAL_HIDDEN_DIM = 128   # hidden dim of LightweightLocalEncoder point_mlp
LOCAL_DIM        = 64    # per-waypoint output dim (early fusion channels)

# ---- VE noise schedule ----
SIGMA_MIN = 0.01
SIGMA_MAX = 10.0
N_LEVELS  = 1000

# ---- CFG ----
P_UNCOND       = 0.1   # probability of dropping ellipsoid conditioning during training
GUIDANCE_SCALE = 1.0   # CFG weight at inference (1.0 = conditional only)

# ---- Training ----
BATCH_SIZE       = 128
LR               = 3e-4
TOTAL_STEPS      = 100_000
EMA_DECAY        = 0.995
EMA_START_STEP   = 1_000
EMA_UPDATE_EVERY = 10

# ---- Sampling ----
N_SAMPLE_STEPS = 25

# ---- Logging ----
LOG_EVERY     = 500
PREVIEW_EVERY = 500
SAVE_EVERY    = 1_000

## 2. Imports

In [11]:
import sys, os, copy, time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, clear_output
import minari

# EllipsoidalCBFSampling/ is the cwd; its parent (SDPMSP_Clone/) must be on path
PROJECT_ROOT = os.path.dirname(os.getcwd())   # SDPMSP_Clone/
NOTEBOOK_DIR = os.getcwd()                    # EllipsoidalCBFSampling/

for p in [PROJECT_ROOT, NOTEBOOK_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from EllipsoidalCBFSampling.models.score_net_ellipsoids import TemporalUnet
from EllipsoidalCBFSampling.models.ve_diffusion_ellipsoids import VEDiffusion
from EllipsoidalCBFSampling.models.samplers_ellipsoids_cfg import dpm_solver_1_cfg_sample

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


## 3. Dataset

In [12]:
class EllipsoidsMinariDataset(Dataset):
    """
    Wraps the `custom/largemaze_ellipsoids-v1` Minari dataset.

    Each item returns:
        traj       : [T_steps, 2]  float32  — x,y waypoints, normalised to [-1, 1]
        ellipsoids : [MAX_ELLIPSOIDS, 4]  float32  — (cx, cy, a, b); zeros for absent ones

    Normalisation uses per-dim min/max computed from the full dataset on first load.
    """

    def __init__(self, dataset_id, local_data_path, max_ellipsoids=5, T_steps=None):
        os.environ['MINARI_DATASETS_PATH'] = os.path.abspath(local_data_path)
        ds = minari.load_dataset(dataset_id)

        trajs      = []   # list of np arrays [T, 2]
        ellipsoids = []   # list of np arrays [max_ellipsoids, 4]

        for ep in ds.iterate_episodes():
            obs = ep.observations  # [T, 2] (or [T, 4]); take only x,y
            trajs.append(obs[:, :2].astype(np.float32))

            centers = np.array(ep.infos['ellipsoids_centers'][0], dtype=np.float32)  # [N, 2]
            radii   = np.array(ep.infos['ellipsoids_radii'][0],   dtype=np.float32)  # [N, 2]
            ell     = np.concatenate([centers, radii], axis=-1)   # [N, 4]

            # Pad / truncate to fixed set size
            N = ell.shape[0]
            if N < max_ellipsoids:
                pad = np.zeros((max_ellipsoids - N, 4), dtype=np.float32)
                ell = np.concatenate([ell, pad], axis=0)
            else:
                ell = ell[:max_ellipsoids]

            ellipsoids.append(ell)

        # Determine horizon
        horizon = T_steps if T_steps is not None else trajs[0].shape[0]
        self.horizon = horizon

        # Truncate / pad each trajectory to the fixed horizon
        fixed_trajs = []
        for t in trajs:
            if t.shape[0] >= horizon:
                fixed_trajs.append(t[:horizon])
            else:
                pad_rows = np.repeat(t[-1:], horizon - t.shape[0], axis=0)
                fixed_trajs.append(np.concatenate([t, pad_rows], axis=0))

        all_traj = np.stack(fixed_trajs, axis=0)   # [N_ep, T, 2]

        # Normalise trajectories to [-1, 1]
        self.xy_min = all_traj.reshape(-1, 2).min(axis=0)   # [2]
        self.xy_max = all_traj.reshape(-1, 2).max(axis=0)   # [2]
        xy_range = self.xy_max - self.xy_min
        xy_range[xy_range < 1e-6] = 1.0   # guard against degenerate dims
        self.xy_range = xy_range

        norm_traj = 2.0 * (all_traj - self.xy_min) / self.xy_range - 1.0  # [-1, 1]

        self.trajs      = torch.tensor(norm_traj,            dtype=torch.float32)
        self.ellipsoids = torch.tensor(np.stack(ellipsoids), dtype=torch.float32)

        print(f"EllipsoidsMinariDataset loaded: {len(self)} episodes, "
              f"horizon={self.horizon}, ellipsoids/ep={max_ellipsoids}")
        print(f"  xy_min={self.xy_min}, xy_max={self.xy_max}")

    def __len__(self):
        return len(self.trajs)

    def __getitem__(self, idx):
        return self.trajs[idx], self.ellipsoids[idx]

    def unnormalize(self, x):
        """Map normalised [-1,1] coords back to world coords. x: [..., 2]"""
        xy_min   = torch.tensor(self.xy_min,   dtype=x.dtype, device=x.device)
        xy_range = torch.tensor(self.xy_range, dtype=x.dtype, device=x.device)
        return (x + 1.0) / 2.0 * xy_range + xy_min

    def normalize(self, x):
        """Map world coords to normalised [-1,1]. x: [..., 2]"""
        xy_min   = torch.tensor(self.xy_min,   dtype=x.dtype, device=x.device)
        xy_range = torch.tensor(self.xy_range, dtype=x.dtype, device=x.device)
        return 2.0 * (x - xy_min) / xy_range - 1.0


ds = EllipsoidsMinariDataset(
    DATASET_ID,
    LOCAL_DATA_PATH,
    max_ellipsoids=MAX_ELLIPSOIDS,
)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
T_STEPS = ds.horizon
print(f'T_STEPS={T_STEPS}, batches per epoch={len(dl)}')

EllipsoidsMinariDataset loaded: 6005 episodes, horizon=64, ellipsoids/ep=5
  xy_min=[-4.9339027 -3.437651 ], xy_max=[4.936099  3.4286566]
T_STEPS=64, batches per epoch=46


## 4. Model Setup

In [13]:
score_net = TemporalUnet(
    state_dim=2,
    T_steps=T_STEPS,
    unet_input_dim=UNET_INPUT_DIM,
    dim_mults=DIM_MULTS,
    local_hidden_dim=LOCAL_HIDDEN_DIM,
    local_dim=LOCAL_DIM,
).to(device)

ve = VEDiffusion(
    model=score_net,
    sigma_min=SIGMA_MIN,
    sigma_max=SIGMA_MAX,
    n_levels=N_LEVELS,
).to(device)

ema_model = copy.deepcopy(score_net).to(device)
for p in ema_model.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.Adam(score_net.parameters(), lr=LR)

n_params = sum(p.numel() for p in score_net.parameters())
print(f'TemporalUnet params: {n_params:,}')

# Sanity check: single forward pass
with torch.no_grad():
    _x  = torch.randn(4, T_STEPS, 2, device=device)
    _s  = torch.ones(4, device=device) * 0.5
    _xs = torch.zeros(4, 2, device=device)
    _xg = torch.ones(4, 2, device=device)
    _el = torch.zeros(4, MAX_ELLIPSOIDS, 4, device=device)
    _out = score_net(_x, _s, _xs, _xg, _el)
    print(f'Forward pass OK: output shape = {tuple(_out.shape)}')

TemporalUnet params: 1,054,486
Forward pass OK: output shape = (4, 64, 2)


## 5. Training + Live Preview

In [ ]:
# ── Random scene helper ──────────────────────────────────────────────────────
# Path setup required by trajectory_generator.py — must run before the training
# loop so that preview() can call generate_random_scene() at each preview step.

_traj_gen_dir = os.path.join(PROJECT_ROOT, 'Diffuser', 'TrajectoryDatasetGeneration')
if _traj_gen_dir not in sys.path:
    sys.path.insert(0, _traj_gen_dir)

_mpd_root = 'C:\\Users\\Owner\\SAFE_DIFFUSION\\mpd-public'
for _p in [_mpd_root, os.path.join(_mpd_root, 'deps', 'torch_robotics')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from trajectory_generator import EnvLargeMazeRandomEllipsoids2D
from torch_robotics.robots import RobotPointMass
from torch_robotics.tasks.tasks import PlanningTask

_tensor_args_cpu = {'device': 'cpu', 'dtype': torch.float32}


def generate_random_scene(max_obstacles=5, dist_factor=0.25):
    """
    Sample a random LargeMaze scene: up to `max_obstacles` ellipsoidal obstacles
    placed in free-space cells, plus a collision-free start/goal pair.

    Returns
    -------
    ell_world : np.ndarray  [max_obstacles, 4]  (cx, cy, a, b), zero-padded, world coords
    start_pos : torch.Tensor [2]  world coords
    goal_pos  : torch.Tensor [2]  world coords
    """
    env   = EnvLargeMazeRandomEllipsoids2D(tensor_args=_tensor_args_cpu,
                                           max_obstacles=max_obstacles)
    robot = RobotPointMass(q_limits=env.limits, tensor_args=_tensor_args_cpu)
    task  = PlanningTask(env=env, robot=robot,
                         obstacle_cutoff_margin=0.08,
                         tensor_args=_tensor_args_cpu)

    min_dist = torch.norm(env.limits[1] - env.limits[0]) * dist_factor

    start_pos = goal_pos = None
    for _ in range(500):
        q_free = task.random_coll_free_q(n_samples=2)
        sp, gp = q_free[0], q_free[1]
        if torch.norm(sp - gp) > min_dist:
            start_pos, goal_pos = sp, gp
            break

    if start_pos is None:   # fallback: use whatever we got last
        start_pos, goal_pos = q_free[0], q_free[1]

    centers = env.ellipsoids_centers   # [N, 2]
    radii   = env.ellipsoids_radii     # [N, 2]
    ell = np.concatenate([centers, radii], axis=-1).astype(np.float32)  # [N, 4]
    N = ell.shape[0]
    if N < max_obstacles:
        ell = np.concatenate([ell, np.zeros((max_obstacles - N, 4), dtype=np.float32)], axis=0)
    else:
        ell = ell[:max_obstacles]

    return ell, start_pos, goal_pos


print("generate_random_scene() ready.")


# ── Fixed preview scenes (generated once at startup) ─────────────────────────

def _generate_scene_with_min_obstacles(max_obstacles, min_obstacles, max_retries=20):
    """Retry until at least min_obstacles non-zero ellipsoids are placed."""
    ell_np = sp = gp = None
    n_nonzero = 0
    for _ in range(max_retries):
        ell_np, sp, gp = generate_random_scene(max_obstacles=max_obstacles)
        n_nonzero = int((np.abs(ell_np).sum(axis=1) > 1e-6).sum())
        if n_nonzero >= min_obstacles:
            return ell_np, sp, gp
    print(f"WARNING: only {n_nonzero} obstacles after {max_retries} retries; using anyway.")
    return ell_np, sp, gp


PREVIEW_SCENES = []
for _i in range(5):
    _ell, _sp, _gp = _generate_scene_with_min_obstacles(
        max_obstacles=MAX_ELLIPSOIDS, min_obstacles=3
    )
    PREVIEW_SCENES.append((
        _ell,                                   # [5, 4]  world coords (numpy, for plotting)
        ds.normalize(_sp.unsqueeze(0)),          # [1, 2]  normalised, CPU
        ds.normalize(_gp.unsqueeze(0)),          # [1, 2]  normalised, CPU
    ))

print(f"Generated {len(PREVIEW_SCENES)} fixed preview scenes.")
for _i, (_e, _s, _g) in enumerate(PREVIEW_SCENES):
    _n = int((np.abs(_e).sum(axis=1) > 1e-6).sum())
    print(f"  scene {_i + 1}: {_n} obstacles")

In [ ]:
import itertools


# ---------------------------------------------------------------------------
# Maze layout (matches EnvLargeMazeRandomEllipsoids2D in trajectory_generator.py)
# ---------------------------------------------------------------------------
_MAZE_MAP = [
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1],
    [1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
    [1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1],
    [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
]


def draw_maze(ax):
    """Draw LargeMaze wall cells as solid gray rectangles and confine axes to maze bounds."""
    for i, row in enumerate(_MAZE_MAP):
        for j, cell in enumerate(row):
            if cell == 1:
                cx = -5.5 + j * 1.0
                cy =  4.0 - i * 1.0
                rect = plt.Rectangle(
                    (cx - 0.5, cy - 0.5), 1.0, 1.0,
                    linewidth=0, facecolor='dimgray', alpha=1.0,
                )
                ax.add_patch(rect)
    ax.set_xlim(-6.0, 6.0)
    ax.set_ylim(-4.5, 4.5)


def draw_ellipsoid(ax, cx, cy, a, b, color='gray', alpha=0.4):
    """Draw a degree-4 superellipse (same formula as MultiEllipsoidField.render)."""
    t = np.linspace(0, 2 * np.pi, 200)
    x = cx + a * np.sign(np.cos(t)) * np.sqrt(np.abs(np.cos(t)))
    y = cy + b * np.sign(np.sin(t)) * np.sqrt(np.abs(np.sin(t)))
    polygon = plt.Polygon(np.column_stack([x, y]),
                          facecolor=color, edgecolor=color, alpha=alpha)
    ax.add_patch(polygon)


def preview(step):
    """Display the 5 fixed preview scenes with denoised trajectories."""
    ema_model.eval()
    fig, axes = plt.subplots(1, len(PREVIEW_SCENES), figsize=(5 * len(PREVIEW_SCENES), 5))
    fig.suptitle(
        f'Step {step:,}  —  DPM-Solver-1 CFG (w={GUIDANCE_SCALE}, {N_SAMPLE_STEPS} steps)',
        fontsize=11,
    )

    with torch.no_grad():
        for j, (ax, (ell_np, xs_pre, xg_pre)) in enumerate(zip(axes, PREVIEW_SCENES)):
            xs_j  = xs_pre.to(device)                                              # [1, 2]
            xg_j  = xg_pre.to(device)                                              # [1, 2]
            ell_j = torch.tensor(ell_np, dtype=torch.float32).unsqueeze(0).to(device)  # [1, N, 4]

            samp = dpm_solver_1_cfg_sample(
                ema_model, ve,
                x_start=xs_j, x_goal=xg_j,
                ellipsoids=ell_j,
                T_steps=T_STEPS,
                n_steps=N_SAMPLE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                device=device,
            )  # [1, T, 2]  normalised

            world = ds.unnormalize(samp.cpu())   # [1, T, 2]  world
            xs_w  = ds.unnormalize(xs_pre)       # [1, 2]  already CPU
            xg_w  = ds.unnormalize(xg_pre)       # [1, 2]  already CPU

            draw_maze(ax)
            for row in ell_np:
                cx, cy, a, b = row
                if a > 1e-4 or b > 1e-4:
                    draw_ellipsoid(ax, cx, cy, a, b)

            ax.plot(world[0, :, 0], world[0, :, 1], lw=1.2, color='steelblue', alpha=0.85)
            ax.scatter(xs_w[0, 0], xs_w[0, 1], c='green', s=60, zorder=5)
            ax.scatter(xg_w[0, 0], xg_w[0, 1], c='red',   s=60, zorder=5)
            ax.set_title(f'fixed scene {j + 1}')
            ax.set_aspect('equal')

    plt.tight_layout()
    display(fig)
    plt.close(fig)
    score_net.train()


def ckpt_path(step):
    """Return the step-stamped checkpoint path under checkpoints/local_feats/."""
    return os.path.join(CKPT_DIR, f've_unet_largemaze_local_feats_step{step:07d}.pt')


# ---- Training loop ----
loader_iter  = itertools.cycle(dl)
loss_history = []
t0           = time.time()

CKPT_DIR = os.path.join(NOTEBOOK_DIR, 'checkpoints', 'local_feats')
os.makedirs(CKPT_DIR, exist_ok=True)

for step in range(1, TOTAL_STEPS + 1):
    score_net.train()

    traj, ellipsoids = next(loader_iter)
    traj       = traj.to(device)        # [B, T, 2]
    ellipsoids = ellipsoids.to(device)  # [B, 5, 4]
    xs = traj[:, 0, :]                  # [B, 2]
    xg = traj[:, -1, :]                 # [B, 2]

    loss, info = ve(traj, xs, xg, ellipsoids, p_uncond=P_UNCOND)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # EMA update
    if step >= EMA_START_STEP and step % EMA_UPDATE_EVERY == 0:
        with torch.no_grad():
            for p_ema, p in zip(ema_model.parameters(), score_net.parameters()):
                p_ema.data.mul_(EMA_DECAY).add_(p.data, alpha=1.0 - EMA_DECAY)

    loss_history.append(info['loss'])

    print("Step: ", step, " Loss: ", loss.item())

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        avg = np.mean(loss_history[-LOG_EVERY:])
        print(f'step {step:>7,} | loss {avg:.4f} | sigma_mean {info["sigma_mean"]:.3f} | {elapsed:.0f}s')

    if step % PREVIEW_EVERY == 0:
        clear_output(wait=True)
        preview(step)

    if step % SAVE_EVERY == 0:
        path = ckpt_path(step)
        torch.save({
            'step':      step,
            'score_net': score_net.state_dict(),
            'ema_model': ema_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'loss_history': loss_history,
            'config': {
                'T_steps':          T_STEPS,
                'unet_input_dim':   UNET_INPUT_DIM,
                'dim_mults':        list(DIM_MULTS),
                'sigma_min':        SIGMA_MIN,
                'sigma_max':        SIGMA_MAX,
                'n_levels':         N_LEVELS,
                'local_hidden_dim': LOCAL_HIDDEN_DIM,
                'local_dim':        LOCAL_DIM,
                'max_ellipsoids':   MAX_ELLIPSOIDS,
                'p_uncond':         P_UNCOND,
                'guidance_scale':   GUIDANCE_SCALE,
                'dataset_id':       DATASET_ID,
                'xy_min':           ds.xy_min.tolist(),
                'xy_max':           ds.xy_max.tolist(),
            },
        }, path)
        print(f'  checkpoint saved → {path}')

print('Training complete.')

## 6. Checkpoint — Save / Load

In [ ]:
# ---- Save manually (if training was interrupted) ----
path = ckpt_path(step)
torch.save({
    'step':      step,
    'score_net': score_net.state_dict(),
    'ema_model': ema_model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'loss_history': loss_history,
    'config': {
        'T_steps':          T_STEPS,
        'unet_input_dim':   UNET_INPUT_DIM,
        'dim_mults':        list(DIM_MULTS),
        'sigma_min':        SIGMA_MIN,
        'sigma_max':        SIGMA_MAX,
        'n_levels':         N_LEVELS,
        'local_hidden_dim': LOCAL_HIDDEN_DIM,
        'local_dim':        LOCAL_DIM,
        'max_ellipsoids':   MAX_ELLIPSOIDS,
        'p_uncond':         P_UNCOND,
        'guidance_scale':   GUIDANCE_SCALE,
        'dataset_id':       DATASET_ID,
        'xy_min':           ds.xy_min.tolist(),
        'xy_max':           ds.xy_max.tolist(),
    },
}, path)
print('Saved →', path)

In [ ]:
# ---- Load ----
# Set LOAD_STEP to the step number of the checkpoint you want to restore.
LOAD_STEP = 10_000   # e.g. 10000, 20000, ...

load_path = ckpt_path(LOAD_STEP)
ckpt = torch.load(load_path, map_location=device, weights_only=False)
cfg  = ckpt['config']

score_net = TemporalUnet(
    state_dim=2,
    T_steps=cfg['T_steps'],
    unet_input_dim=cfg['unet_input_dim'],
    dim_mults=tuple(cfg['dim_mults']),
    local_hidden_dim=cfg['local_hidden_dim'],
    local_dim=cfg['local_dim'],
).to(device)

ve = VEDiffusion(
    score_net,
    sigma_min=cfg['sigma_min'],
    sigma_max=cfg['sigma_max'],
    n_levels=cfg['n_levels'],
).to(device)

ema_model = copy.deepcopy(score_net).to(device)
score_net.load_state_dict(ckpt['score_net'])
ema_model.load_state_dict(ckpt['ema_model'])
ema_model.eval()

T_STEPS        = cfg['T_steps']
GUIDANCE_SCALE = cfg.get('guidance_scale', 3.0)
MAX_ELLIPSOIDS = cfg.get('max_ellipsoids', 5)

print(f"Loaded step {ckpt['step']:,}  |  T_steps={T_STEPS}  |  {load_path}")

## 7. Final Sampling — Random Scenes

In [ ]:
N_EVAL = 100  # total panels: 25 rows × 4 cols

# ── Generate N_EVAL random scenes ────────────────────────────────────────────
print("Generating random scenes...")
rand_ells  = []   # list of np.ndarray [MAX_ELLIPSOIDS, 4]  world coords
rand_xs_w  = []   # list of torch.Tensor [2]  world coords
rand_xg_w  = []   # list of torch.Tensor [2]  world coords

for i in range(N_EVAL):
    ell, sp, gp = generate_random_scene(max_obstacles=MAX_ELLIPSOIDS)
    rand_ells.append(ell)
    rand_xs_w.append(sp)
    rand_xg_w.append(gp)
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{N_EVAL} scenes generated")

rand_ells_t = torch.tensor(np.stack(rand_ells), dtype=torch.float32)   # [N_EVAL, 5, 4]  world
rand_xs     = ds.normalize(torch.stack(rand_xs_w))                     # [N_EVAL, 2]  normalised
rand_xg     = ds.normalize(torch.stack(rand_xg_w))                     # [N_EVAL, 2]  normalised

# ── Sample and plot ───────────────────────────────────────────────────────────
print("Sampling trajectories...")
ema_model.eval()

fig, axes = plt.subplots(25, 4, figsize=(20, int(10 / 2 * 25)))
fig.suptitle(
    f'DPM-Solver-1 CFG  (w={GUIDANCE_SCALE}, {N_SAMPLE_STEPS} steps)  —  Random Scenes',
    fontsize=13,
)

for j, ax in enumerate(axes.flat):
    xs_j  = rand_xs[j:j+1].to(device)       # [1, 2]  normalised
    xg_j  = rand_xg[j:j+1].to(device)       # [1, 2]  normalised
    ell_j = rand_ells_t[j:j+1].to(device)   # [1, 5, 4]  world coords

    samp = dpm_solver_1_cfg_sample(
        ema_model, ve,
        x_start=xs_j, x_goal=xg_j,
        ellipsoids=ell_j,
        T_steps=T_STEPS,
        n_steps=N_SAMPLE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        device=device,
    )  # [1, T, 2]  normalised

    world = ds.unnormalize(samp.cpu())   # [1, T, 2]  world
    xs_w  = ds.unnormalize(xs_j.cpu())   # [1, 2]  world
    xg_w  = ds.unnormalize(xg_j.cpu())   # [1, 2]  world

    draw_maze(ax)
    for row in rand_ells[j]:
        cx, cy, a, b = row
        if a > 1e-4 or b > 1e-4:
            draw_ellipsoid(ax, cx, cy, a, b)

    ax.plot(world[0, :, 0], world[0, :, 1], lw=1.4, color='steelblue', alpha=0.9)
    ax.scatter(xs_w[0, 0], xs_w[0, 1], c='green', s=60, zorder=6, label='start')
    ax.scatter(xg_w[0, 0], xg_w[0, 1], c='red',   s=60, zorder=6, label='goal')
    ax.set_aspect('equal')
    if j == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── 100 samples of a single randomly generated scene ─────────────────────────
print("Generating random scene for 100-sample test...")
ell_rand, sp_rand, gp_rand = generate_random_scene(max_obstacles=MAX_ELLIPSOIDS)

xs_tc  = ds.normalize(sp_rand.unsqueeze(0)).to(device)                      # [1, 2]  normalised
xg_tc  = ds.normalize(gp_rand.unsqueeze(0)).to(device)                      # [1, 2]  normalised
ell_tc = torch.tensor(ell_rand, dtype=torch.float32).unsqueeze(0).to(device) # [1, 5, 4]  world

N_REPEAT = 100

ema_model.eval()
samples = []
with torch.no_grad():
    for _ in range(N_REPEAT):
        samp = dpm_solver_1_cfg_sample(
            ema_model, ve,
            x_start=xs_tc, x_goal=xg_tc,
            ellipsoids=ell_tc,
            T_steps=T_STEPS,
            n_steps=N_SAMPLE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            device=device,
        )  # [1, T, 2]  normalised
        samples.append(ds.unnormalize(samp.cpu()))   # [1, T, 2]  world

xs_w_plot = ds.unnormalize(xs_tc.cpu())   # [1, 2]
xg_w_plot = ds.unnormalize(xg_tc.cpu())   # [1, 2]

fig, axes = plt.subplots(10, 10, figsize=(25, 25))
fig.suptitle(
    f'Random scene  —  {N_REPEAT} independent samples\n'
    f'DPM-Solver-1 CFG  (w={GUIDANCE_SCALE}, {N_SAMPLE_STEPS} steps)',
    fontsize=13,
)

for k, ax in enumerate(axes.flat):
    draw_maze(ax)
    for row in ell_rand:
        cx, cy, a, b = row
        if a > 1e-4 or b > 1e-4:
            draw_ellipsoid(ax, cx, cy, a, b)
    world_k = samples[k]
    ax.plot(world_k[0, :, 0], world_k[0, :, 1], lw=1.2, color='steelblue', alpha=0.85)
    ax.scatter(xs_w_plot[0, 0], xs_w_plot[0, 1], c='green', s=40, zorder=6)
    ax.scatter(xg_w_plot[0, 0], xg_w_plot[0, 1], c='red',   s=40, zorder=6)
    ax.set_aspect('equal')
    ax.set_title(f'sample {k + 1}', fontsize=7)

plt.tight_layout()
plt.show()

## 8. Loss Curve

In [ ]:
smooth = np.convolve(loss_history, np.ones(200) / 200, mode='valid')
plt.figure(figsize=(10, 3))
plt.plot(loss_history, alpha=0.2, color='steelblue', linewidth=0.5)
plt.plot(np.arange(len(smooth)) + 100, smooth, color='steelblue', linewidth=1.5)
plt.xlabel('Step')
plt.ylabel('DSM Loss')
plt.title('Training Loss — LargeMaze Ellipsoids Diffuser (CFG)')
plt.tight_layout()
plt.show()